# M4 Training from Scratch - 3-Layer Strategy to Fix 6→0 Error

**⚠️ NOT Fine-tuning - Training from Scratch with MobileNetV1**

**Target**: Reduce 6→0 error rate from 20.4% to <5%

## 🎯 3-Layer Strategy

### Layer 1: WeightedRandomSampler - Balance "Diet" 🍽️
- Force AI to see rare digits (6,7,8,9) equally with common digits (0,1,2)
- Prevents AI from "guessing" 0 for high probability

### Layer 2: Sharpening Pre-processing - "Magnifying Glass" 🔍
- Apply Unsharp Masking + CLAHE to blurry images
- Enhances blurry 6→0 features (the "hook" on top of 6)

### Layer 3: Weighted Loss - Increase "Punishment" ⚖️
- 📊 **Optimized weights based on failure analysis:**
  - Digit 6: **4.0x** (Tỉ lệ lỗi 26% - CAO NHẤT) - 8x vs digit 0
  - Digit 1: **3.5x** (Số lượng lỗi tuyệt đối nhiều nhất: 370 lần)
  - Digit 8: **3.0x** (Hay bị nhầm thành 0 và 1)
  - Digit 9: **2.5x** (Dữ liệu ít, dễ bị bỏ qua)
  - Digits 4,5,7: **2.0x** (Hay nhầm chéo)
  - Digits 2,3: **1.5x** (Tương đối ổn định)
  - Digit 0: **0.5x** (GIẢM TRỌNG SỐ - AI bớt đoán mò)

---

**Expected Results:**
- 6→0 error rate: 20.4% → <5%
- 1→0 error rate: 19.2% → <5%
- 8→0 error rate: 9.7% → <3%
- Overall pipeline accuracy: 98-99%
- Training time: ~30-45 minutes on GPU T4

## 📦 Setup - Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio -q
!pip install opencv-python-headless -q
!pip install tqdm pandas -q

import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter
import json

print("✅ All dependencies installed!")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## 📂 Prepare Data - Copy from Drive

In [ ]:
# Define paths
DATA_DIR = "/content/data_4digit2"
LABELS_FILE = "/content/images_4digit2.csv"
OUTPUT_DIR = "/content/trained_models"

# Update these paths to match your Google Drive structure
DRIVE_DATA_DIR = "/content/drive/MyDrive/Project/data/data_4digit2"
DRIVE_LABELS_FILE = "/content/drive/MyDrive/Project/data/images_4digit2.csv"

# Create directories
!mkdir -p {DATA_DIR}
!mkdir -p {OUTPUT_DIR}

print("📂 Copying data from Google Drive...")
print(f"  From: {DRIVE_DATA_DIR}")
print(f"  To: {DATA_DIR}")

# Copy data (uncomment if you want to copy data to Colab)
# !cp -r {DRIVE_DATA_DIR}/* {DATA_DIR}/
# !cp {DRIVE_LABELS_FILE} {LABELS_FILE}

# Alternative: Use direct paths to Drive
DATA_DIR = DRIVE_DATA_DIR
LABELS_FILE = DRIVE_LABELS_FILE

print(f"\n✅ Data ready!")
print(f"  Images: {DATA_DIR}")
print(f"  Labels: {LABELS_FILE}")
print(f"  Output: {OUTPUT_DIR}")

## 🔧 Define CRNN Model Architecture

In [ ]:
class CRNN(nn.Module):
    """CRNN Model for Meter Reading"""
    def __init__(self, num_chars=11, num_channels=1, img_height=64, hidden_size=256):
        super().__init__()
        self.num_chars = num_chars
        self.hidden_size = hidden_size

        # CNN Feature Extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(num_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),

            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
        )

        h_out = img_height // 16
        self.rnn_input_size = 512 * h_out

        # Bidirectional LSTM
        self.rnn = nn.LSTM(
            input_size=self.rnn_input_size,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(hidden_size * 2, num_chars)

    def forward(self, x):
        conv_out = self.cnn(x)
        b, c, h, w = conv_out.size()
        features = conv_out.permute(0, 3, 1, 2).contiguous().view(b, w, c * h)
        rnn_out, _ = self.rnn(features)
        logits = self.fc(rnn_out).permute(1, 0, 2)
        return logits

print("✅ CRNN model architecture defined!")

## 🎯 Layer 3: Weighted Loss - Higher Punishment for Digit 6 ⚖️

In [ ]:
class DigitWeightedLoss(nn.Module):
    """
    Layer 3: Weighted Loss - Higher punishment for digit 6 errors
    
    📊 BẢNG TRỌNG SỐ ĐỀ XUẤT (Dựa trên phân tích lỗi thực tế)
    
    Chữ số | Trọng số | Lý do
    --------|----------|-------
      6    |   4.0    | Tỉ lệ lỗi 26% (CAO NHẤT) - Target chính!
      1    |   3.5    | Số lượng lỗi tuyệt đối nhiều nhất (370 lần)
      8    |   3.0    | Hay bị nhầm thành 0 và 1
      9    |   2.5    | Dữ liệu ít, dễ bị mô hình bỏ qua
      7    |   2.0    | Hay nhầm với 1
      5    |   2.0    | Hay nhầm thành 8 (mirror)
      4    |   2.0    | Hay nhầm thành 3
      2    |   1.5    | Tương đối ổn định
      3    |   1.5    | Tương đối ổn định
      0    |   0.5    | GIẢM TRỌNG SỐ - AI bớt "đoán mò" là 0
    
    Note: Digit 6 có trọng số 8x cao hơn digit 0!
    """
    def __init__(self, digit_weights=None, blank_idx=10):
        super().__init__()
        if digit_weights is None:
            # 📊 TRỌNG SỐ ĐỀ XUẤT - Dựa trên phân tích lỗi thực tế
            self.digit_weights = torch.tensor([
                0.5,  # 0: GIẢM TRỌNG SỐ - quá phổ biến, AI hay đoán mò
                3.5,  # 1: Số lượng lỗi tuyệt đối nhiều nhất (370 lần)
                1.5,  # 2: Tương đối ổn định
                1.5,  # 3: Tương đối ổn định
                2.0,  # 4: Hay nhầm thành 3
                2.0,  # 5: Hay nhầm thành 8
                4.0,  # 6: TARGET! Tỉ lệ lỗi 26% (cao nhất) - 8x vs 0
                2.0,  # 7: Hay nhầm thành 1
                3.0,  # 8: Hay bị nhầm thành 0 và 1
                2.5,  # 9: Dữ liệu ít, dễ bị bỏ qua
                1.0,  # 10: Blank (CTC)
            ])
        else:
            self.digit_weights = torch.tensor(digit_weights)

        self.blank_idx = blank_idx
        self.ctc_loss = nn.CTCLoss(blank=blank_idx, reduction='none')

    def forward(self, logits, targets, input_lengths, target_lengths):
        """
        Args:
            logits: (T, N, C) - T time steps, N batch, C classes
            targets: Concatenated target labels
            input_lengths: Length of each sequence
            target_lengths: Length of each target

        Returns:
            Weighted loss
        """
        log_probs = logits.log_softmax(2)

        # Calculate base CTC loss (per sample)
        losses = []
        for i in range(logits.size(1)):
            target_i = targets[
                sum(target_lengths[:i]):sum(target_lengths[:i+1])
            ]
            loss_i = self.ctc_loss(
                log_probs[:, i:i+1, :],
                target_i.unsqueeze(0),
                input_lengths[i:i+1],
                target_lengths[i:i+1]
            )

            # Apply weight based on target digits
            weights = self.digit_weights[target_i]
            avg_weight = weights.mean().to(logits.device)

            # Weighted loss: higher weight for rare digits
            weighted_loss = loss_i * avg_weight
            losses.append(weighted_loss)

        return torch.stack(losses).mean()

# Initialize weighted loss with OPTIMIZED weights
criterion = DigitWeightedLoss()
print("✅ Layer 3: Weighted Loss initialized!")
print("   📊 Digit weights (based on failure analysis):")
for i, w in enumerate(criterion.digit_weights):
    if i < 10:
        print(f"     Digit {i}: {w.item():.1f}x", end="")
        if i == 6:
            print(" ← TARGET! (8x vs 0)")
        elif i == 0:
            print(" (reduced - AI stops guessing)")
        else:
            print()
print()

## 🔍 Layer 2: Sharpening Pre-processing - "Magnifying Glass"

In [ ]:
class SharpenTransform:
    """
    Layer 2: Sharpening Pre-processing - "Magnifying Glass" for blurry 6→0
    
    Techniques:
    1. Unsharp Masking: Enhance edges and fine details
    2. CLAHE: Improve local contrast
    
    Target: Make the "hook" on top of 6 more visible
    """
    def __init__(self, sharpen_strength=1.5, clahe_clip=2.0):
        self.sharpen_strength = sharpen_strength
        self.clahe_clip = clahe_clip

    def __call__(self, image: np.ndarray) -> np.ndarray:
        # Convert to grayscale if needed
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        # Step 1: Unsharp Masking (Sharpening)
        # Create a blurred version
        blurred = cv2.GaussianBlur(gray, (0, 0), 3)
        # Sharpen: original + strength * (original - blurred)
        sharpened = cv2.addWeighted(
            gray,
            1 + self.sharpen_strength,
            blurred,
            -self.sharpen_strength,
            0
        )

        # Step 2: CLAHE (Contrast Limited Adaptive Histogram Equalization)
        clahe = cv2.createCLAHE(clipLimit=self.clahe_clip, tileGridSize=(4, 4))
        enhanced = clahe.apply(sharpened)

        return enhanced

# Initialize sharpening transform
sharpen_transform = SharpenTransform(sharpen_strength=1.5, clahe_clip=2.0)
print("✅ Layer 2: Sharpening transform initialized!")
print(f"   Sharpen strength: 1.5")
print(f"   CLAHE clip limit: 2.0")

## 📊 Layer 1: Dataset with WeightedRandomSampler - Balance "Diet" 🍽️

In [ ]:
class MeterDataset(Dataset):
    """
    Dataset with digit-based sampling strategy
    
    Layer 1: Samples with rare digits (6,7,8,9) get higher weights
    This forces AI to see rare digits equally with common digits (0,1,2)
    """
    def __init__(self, images_dir, labels_file, transform=None, sharpen=None):
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.sharpen = sharpen

        # Load labels
        self.labels_df = pd.read_csv(labels_file)
        self.samples = []

        # Calculate digit frequencies
        digit_counts = Counter()
        for _, row in self.labels_df.iterrows():
            value = str(row['value'])
            for digit in value:
                if digit.isdigit():
                    digit_counts[int(digit)] += 1

        print("Digit distribution in dataset:")
        total_digits = sum(digit_counts.values())
        for digit in sorted(digit_counts.keys()):
            count = digit_counts[digit]
            print(f"  Digit {digit}: {count:5d} occurrences ({count/total_digits*100:5.2f}%)")

        # Calculate sample weights (inverse of digit frequency)
        # Samples with rare digits get higher weights
        self.sample_weights = []

        for _, row in self.labels_df.iterrows():
            value = str(row['value'])

            # Calculate weight based on rarest digit in this sample
            min_freq = float('inf')
            for digit in value:
                if digit.isdigit():
                    freq = digit_counts[int(digit)]
                    min_freq = min(min_freq, freq)

            # Inverse frequency: rarer digits = higher weight
            weight = 1.0 / (min_freq + 1e-6)
            self.sample_weights.append(weight)

        # Normalize weights
        total_weight = sum(self.sample_weights)
        self.sample_weights = [w / total_weight for w in self.sample_weights]

        print(f"\nSample weight range: {min(self.sample_weights):.6f} - {max(self.sample_weights):.6f}")

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        img_path = self.images_dir / row['photo_name']

        # Load image (black digit crop from M3.5)
        # For fine-tuning, use M3.5 black digit crops
        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f"Cannot load image: {img_path}")

        # Apply sharpening (Layer 2)
        if self.sharpen is not None:
            image = self.sharpen(image)
        else:
            # Convert to grayscale
            if len(image.shape) == 3:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Resize
        image = cv2.resize(image, (224, 64))

        # Normalize
        image = image.astype(np.float32) / 255.0
        image = (image - 0.5) / 0.5

        # Get label
        label = str(row['value'])

        # Convert label to indices
        label_indices = [int(d) for d in label if d.isdigit()]

        return {
            'image': torch.from_numpy(image).unsqueeze(0).float(),
            'label': torch.tensor(label_indices, dtype=torch.long),
            'label_length': len(label_indices),
            'text': label
        }

print("✅ Layer 1: MeterDataset with WeightedRandomSampler defined!")

## 🔄 Helper Functions

In [ ]:
def collate_fn(batch):
    """
    Custom collate function for variable-length sequences
    
    IMPORTANT: Calculate input_lengths correctly based on CNN architecture!
    - Input width: 224
    - CNN width reduction: 2 × 2 × 2 × 1 = 8
    - Output width: 224 / 8 = 28 time steps
    """
    images = torch.stack([item['image'] for item in batch])
    labels = torch.cat([item['label'] for item in batch])
    label_lengths = torch.tensor([item['label_length'] for item in batch])
    
    # ✅ FIX: Correct input_lengths = actual output width from CNN
    # With input width 224 and CNN reduction factor 8: 224/8 = 28
    input_lengths = torch.tensor([28] * len(batch), dtype=torch.long)

    return {
        'images': images,
        'labels': labels,
        'input_lengths': input_lengths,
        'label_lengths': label_lengths,
        'texts': [item['text'] for item in batch]
    }

print("✅ Helper functions defined!")
print("   ⚠️  input_lengths = 28 (calculated from CNN architecture)")

## 🚀 Load Model & Data

⚠️ **TRAINING FROM SCRATCH - NO PRE-TRAINED WEIGHTS**

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# ⚠️ TRAINING FROM SCRATCH - NO PRE-TRAINED WEIGHTS
print("="*70)
print("⚠️  TRAINING FROM SCRATCH - NO PRE-TRAINED WEIGHTS")
print("="*70)
print()
print("Creating fresh model with CRNN architecture...")
print("This will take LONGER but give BETTER results for YOUR data!")
print()

# Create model from scratch - NO pre-trained weights
model = CRNN(num_chars=11, img_height=64, hidden_size=256).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model created from scratch!")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB")
print()

# 🔍 VERIFY ARCHITECTURE - CTC Alignment Check
print("="*70)
print("🔍 ARCHITECTURE VERIFICATION - CTC Alignment")
print("="*70)

# Test forward pass to verify output dimensions
test_input = torch.randn(1, 1, 64, 224).to(device)  # Batch=1, Channel=1, H=64, W=224
with torch.no_grad():
    test_output = model(test_input)

output_length = test_output.size(0)  # T (time steps)
num_classes = test_output.size(2)    # C (classes)

print(f"\n📏 Input Dimensions:")
print(f"   Height: 64")
print(f"   Width: 224")
print(f"   Expected sequence length: 4 digits")

print(f"\n📐 CNN Architecture:")
print(f"   Pool1: stride (2,2) → H:64→32, W:224→112")
print(f"   Pool2: stride (2,2) → H:32→16, W:112→56")
print(f"   Pool3: stride (2,2) → H:16→8,  W:56→28")
print(f"   Pool4: stride (2,1) → H:8→4,  W:28→28")
print(f"   Width reduction factor: 8")

print(f"\n✅ Output Dimensions:")
print(f"   Time steps (T): {output_length}")
print(f"   Batch size (N): 1")
print(f"   Num classes (C): {num_classes}")

print(f"\n🎯 CTC Alignment Check:")
print(f"   Required for 4 digits: T ≥ 2×4+1 = 9")
print(f"   Actual T: {output_length}")
if output_length >= 9:
    print(f"   ✅ PASS - Sufficient time steps for CTC!")
else:
    print(f"   ❌ FAIL - Need T ≥ 9, got {output_length}")

print(f"\n📋 CTC Configuration:")
print(f"   Blank index: 10")
print(f"   Digit classes: 0-9 (10 classes)")
print(f"   Total classes: {num_classes} (10 digits + 1 blank)")
print(f"   input_lengths: {output_length} (MUST match output T)")
print("="*70)
print()

# Load dataset
print("[Loading Dataset with Layer 1: WeightedRandomSampler]")
dataset = MeterDataset(
    images_dir=DATA_DIR,
    labels_file=LABELS_FILE,
    sharpen=sharpen_transform  # Layer 2: Sharpening
)
print()

# Split train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Layer 1: Create WeightedRandomSampler for training
train_sampler = WeightedRandomSampler(
    weights=[dataset.sample_weights[i] for i in train_dataset.indices],
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=train_sampler,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Val samples: {len(val_dataset)}")
print(f"✅ Layer 1: WeightedRandomSampler applied - rare digits will be seen more often!")

# Verify one batch to ensure input_lengths is correct
print("\n🔍 Verifying batch dimensions...")
sample_batch = next(iter(train_loader))
print(f"   Batch images shape: {sample_batch['images'].shape}")
print(f"   Batch input_lengths: {sample_batch['input_lengths'][0].item()}")
print(f"   Expected output T: {output_length}")
if sample_batch['input_lengths'][0].item() == output_length:
    print(f"   ✅ VERIFIED - input_lengths matches output T!")
else:
    print(f"   ❌ MISMATCH - input_lengths != output T")

## 🏋️ Training & Validation Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc="Training")
    for batch in pbar:
        images = batch['images'].to(device)
        labels = batch['labels'].to(device)
        input_lengths = batch['input_lengths'].to(device)
        label_lengths = batch['label_lengths'].to(device)

        # Forward pass
        logits = model(images)

        # Calculate loss
        loss = criterion(logits, labels, input_lengths, label_lengths)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calculate accuracy (greedy decoding)
        preds = logits.argmax(dim=2)
        for i in range(preds.size(1)):
            pred_seq = preds[:, i]
            pred_text = ''.join([str(p.item()) if p.item() < 10 else '' for p in pred_seq])

            # Remove consecutive duplicates
            pred_text_clean = []
            last_char = None
            for c in pred_text:
                if c != last_char:
                    pred_text_clean.append(c)
                    last_char = c

            pred_text_clean = ''.join(pred_text_clean)

            if pred_text_clean == batch['texts'][i]:
                total_correct += 1
            total_samples += 1

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{total_correct/total_samples:.4f}'})

    return total_loss / len(dataloader), total_correct / total_samples


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    # Digit-wise accuracy
    digit_correct = 0
    digit_total = 0

    # Track 6→0 errors
    six_to_zero_errors = 0
    total_six = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            input_lengths = batch['input_lengths'].to(device)
            label_lengths = batch['label_lengths'].to(device)

            # Forward pass
            logits = model(images)

            # Calculate loss
            loss = criterion(logits, labels, input_lengths, label_lengths)
            total_loss += loss.item()

            # Calculate accuracy
            preds = logits.argmax(dim=2)
            for i in range(preds.size(1)):
                pred_seq = preds[:, i]
                pred_text = ''.join([str(p.item()) if p.item() < 10 else '' for p in pred_seq])

                # Remove consecutive duplicates
                pred_text_clean = []
                last_char = None
                for c in pred_text:
                    if c != last_char:
                        pred_text_clean.append(c)
                        last_char = c

                pred_text_clean = ''.join(pred_text_clean)
                true_text = batch['texts'][i]

                if pred_text_clean == true_text:
                    total_correct += 1
                total_samples += 1

                # Digit-wise accuracy & 6→0 tracking
                for t, p in zip(true_text, pred_text_clean):
                    if t == '6':
                        total_six += 1

                    if t == p:
                        digit_correct += 1
                    digit_total += 1

                    # Track 6→0 errors
                    if t == '6' and p == '0':
                        six_to_zero_errors += 1

    metrics = {
        'loss': total_loss / len(dataloader),
        'accuracy': total_correct / total_samples,
        'digit_accuracy': digit_correct / digit_total if digit_total > 0 else 0,
        'six_to_zero_error_rate': six_to_zero_errors / total_six if total_six > 0 else 0
    }

    return metrics

print("✅ Training & validation functions defined!")

## 🎯 Start Training!

⚠️ **TRAINING FROM SCRATCH - Using MobileNetV1 architecture**

In [ ]:
# Configuration
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3  # Standard LR for training from scratch

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"  Standard LR for training from scratch")
print()

print("="*70)
print("STARTING TRAINING FROM SCRATCH WITH 3-LAYER STRATEGY")
print("="*70)
print("\n✅ Architecture: CRNN with custom CNN")
print("✅ NO pre-trained weights - fresh initialization")
print("\nLayer 1: WeightedRandomSampler - Force AI to see rare digits equally")
print("Layer 2: Sharpening - Enhance blurry 6→0 features")
print("Layer 3: Weighted Loss - 4x punishment for digit 6 errors")
print("="*70)
print()

best_val_acc = 0
best_six_to_zero_rate = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")

    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    val_metrics = validate(model, val_loader, criterion, device)

    print(f"\n{'='*70}")
    print(f"RESULTS:")
    print(f"{'='*70}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_metrics['loss']:.4f} | Val   Acc: {val_metrics['accuracy']:.4f}")
    print(f"\n📊 Digit Accuracy: {val_metrics['digit_accuracy']:.2%}")
    print(f"🎯 6→0 Error Rate: {val_metrics['six_to_zero_error_rate']:.2%} ← TARGET!")

    # Save best model (by accuracy)
    if val_metrics['accuracy'] > best_val_acc:
        best_val_acc = val_metrics['accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_metrics['accuracy'],
            'val_loss': val_metrics['loss'],
            'six_to_zero_error_rate': val_metrics['six_to_zero_error_rate'],
        }, f'{OUTPUT_DIR}/best_model.pth')
        print(f"\n✅ Best model saved (Val Acc: {best_val_acc:.4f})")

    # Save best model (by 6→0 error rate)
    if val_metrics['six_to_zero_error_rate'] < best_six_to_zero_rate:
        best_six_to_zero_rate = val_metrics['six_to_zero_error_rate']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_metrics['accuracy'],
            'val_loss': val_metrics['loss'],
            'six_to_zero_error_rate': val_metrics['six_to_zero_error_rate'],
        }, f'{OUTPUT_DIR}/best_six_to_zero_model.pth')
        print(f"✅ Best 6→0 model saved (Error rate: {best_six_to_zero_rate:.2%})")

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"\n📈 Final Results:")
print(f"   Best Val Accuracy: {best_val_acc:.2%}")
print(f"   Best 6→0 Error Rate: {best_six_to_zero_rate:.2%}")
print(f"\n📁 Models saved to: {OUTPUT_DIR}")
print(f"   - best_model.pth: Best overall accuracy")
print(f"   - best_six_to_zero_model.pth: Best 6→0 error rate")

## 📤 Copy Trained Models Back to Drive

In [ ]:
# Copy models back to Google Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Project/model/M4_trained"
!mkdir -p {DRIVE_OUTPUT_DIR}

print("Copying trained models to Google Drive...")
!cp {OUTPUT_DIR}/best_model.pth {DRIVE_OUTPUT_DIR}/
!cp {OUTPUT_DIR}/best_six_to_zero_model.pth {DRIVE_OUTPUT_DIR}/

print(f"\n✅ Models saved to: {DRIVE_OUTPUT_DIR}")
print("\nYou can now use these trained models in your pipeline!")

## 🎯 Summary - Training Complete

### ✅ 3-Layer Strategy Applied:

**Layer 1: WeightedRandomSampler** 🍽️
- Balanced "diet" - AI now sees rare digits (6,7,8,9) equally with common digits
- Can no longer "guess" 0 for easy probability

**Layer 2: Sharpening Pre-processing** 🔍
- Applied Unsharp Masking + CLAHE to enhance blurry images
- Made the "hook" on top of 6 more visible

**Layer 3: Weighted Loss** ⚖️
- 📊 **Optimized weights based on actual failure analysis:**
  - Digit 6: **4.0x** (8x higher than digit 0) - Target chính!
  - Digit 1: **3.5x** (370 errors - cao nhất về số lượng)
  - Digit 8: **3.0x** (nhầm 0 và 1 nhiều)
  - Digit 9: **2.5x** (ít dữ liệu,容易被 bỏ qua)
  - Digits 4,5,7: **2.0x** (hay nhầm chéo)
  - Digits 2,3: **1.5x** (ổn định nhưng cần cải thiện)
  - Digit 0: **0.5x** (giảm trọng - AI ngừng đoán mò)

### 🔧 CRITICAL FIX - CTC Alignment Issue

**❌ Original Problem:** `input_lengths = 160` but actual CNN output = 28 time steps
- This caused CTC Loss to read beyond tensor boundaries
- Result: Loss = NaN/Inf, Accuracy = 0%

**✅ Solution:**
1. Corrected `input_lengths` to match CNN output: `28`
2. Added architecture verification cell to confirm dimensions
3. CTC now properly aligned with model output

**📊 Architecture:**
```
Input:  (1, 64, 224)  →  Width = 224
CNN:   4 pooling layers → Width reduction = 8
Output: (28, N, 11)    →  Time steps = 28

CTC Requirement: T ≥ 2N + 1 = 2×4+1 = 9
Actual T: 28 ✅ PASS
```

### 📈 Training Type: **FROM SCRATCH**
- ✅ **NOT Fine-tuning** - Training from scratch
- ✅ **CRNN Architecture** - Custom CNN + BiLSTM
- ✅ **3-Layer Strategy** - Comprehensive approach
- ✅ **No Pre-trained Weights** - Fresh initialization
- ✅ **Fixed CTC Alignment** - input_lengths corrected

### 📊 Why Train Acc = 0 Initially?

Before the fix, Acc = 0 was caused by **CTC misalignment**, not:
- ❌ Learning rate too high
- ❌ Not enough epochs
- ❌ Poor initialization

The fix ensures CTC Loss can compute correctly, allowing the model to learn.

### 🎯 Expected Results:
- **6→0 Error Rate**: 20.4% → <5%
- **1→0 Error Rate**: 19.2% → <5%
- **8→0 Error Rate**: 9.7% → <3%
- **Overall OCR Accuracy**: 76.55% → ~85-90%
- **Pipeline Success**: 96.25% → ~98-99%

### 🚀 Next Steps:
1. Copy trained models to your local machine
2. Update pipeline script: `M4_MODEL = "M4_trained/best_six_to_zero_model.pth"`
3. Run full pipeline to verify improvements
4. Analyze new failure patterns (if any)

---

**Training Time**: ~30-45 minutes on GPU T4
**Cost**: Free on Colab 💰

### 📊 Why These Weights?

Các trọng số này được tính toán dựa trên phân tích 1,473 lỗi thực tế:

1. **Digit 6 = 4.0x**: 
   - Tỉ lệ lỗi cao nhất (26%)
   - Khi sai, ảnh hưởng lớn nhất (thay đổi giá trị đọc)

2. **Digit 1 = 3.5x**:
   - Số lượng lỗi nhiều nhất (370 lần)
   - Mặc dù tỉ lệ lỗi không cao nhất, nhưng xuất hiện rất nhiều

3. **Digit 8 = 3.0x**:
   - Hay bị nhầm thành cả 0 và 1
   - Cần học kỹ đặc trưng "hai vòng lặp"

4. **Digit 0 = 0.5x**:
   - GIẢM trọng số - AI không được đoán mò
   - Buộc phải học đặc trưng thật sự, không dựa vào probability

### 🎓 Key Insight

> "AI không học từ dữ liệu cân bằng, nó học từ những gì nó thấy nhiều nhất.
>  giảm trọng số 0 và tăng trọng số 6 sẽ buộc AI tập trung vào chất lượng thay vì số lượng."

### 🐛 Debug Tip

If you still see Acc = 0 after several epochs:
1. Check `input_lengths` matches CNN output width
2. Verify blank index = 10 (not conflicting with digits)
3. Ensure CTC loss is not NaN/Inf
4. Confirm digit labels are in range [0, 9]